# Proyecto Innovacien — 1. Obtención y limpieza de datos

**Curso:** Código y Programación · Samsung Innovation Campus Chile 2026 — Cohort 2

## Pregunta de análisis

> **¿Cómo se distribuyen en Chile los registros de las tres especies invasoras
> que el modelo identifica —jabalí, liebre europea y rata gris— y en qué se
> diferencian sus patrones territoriales?**

El proyecto tiene un modelo de visión (`core/best.pt`, YOLO11s) entrenado para
reconocer **tres especies**, y solo tres:

| Etiqueta del modelo | Nombre común | Nombre científico |
|---|---|---|
| `jabali` | Jabalí | *Sus scrofa* |
| `liebre` | Liebre europea | *Lepus europaeus* |
| `rata gris` | Rata gris | *Rattus norvegicus* |

Todo el análisis se limita a estas tres. Esa restricción es lo que hace que el
análisis y el producto hablen de lo mismo: la aplicación no identifica nada que
el análisis no cubra, ni al revés.

Cada especie se consulta por su **`speciesKey`** de GBIF, un identificador
numérico estable. Buscar por nombre científico obligaría a resolver sinónimos
taxonómicos a mano; el `speciesKey` evita ese problema de origen.

### Fuente única

**GBIF** — Global Biodiversity Information Facility (`api.gbif.org`). Los
registros chilenos vienen de monitoreo institucional, ciencia ciudadana
(iNaturalist) y colecciones de museo. Licencias CC0 / CC-BY / CC-BY-NC.

## Preparación del entorno

El notebook vive en `notebooks/`, pero el código y los datos están en la raíz.
Ajustamos el directorio de trabajo para que las rutas funcionen igual desde
cualquiera de los dos lugares.

In [1]:
import os
import sys
from datetime import date
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)

print("Raíz del proyecto:", RAIZ)
print("Fecha de ejecución:", date.today().isoformat())

Raíz del proyecto: /Users/martindroguett/Desktop/Innovacien/Proyecto-Samsung-Innovacien
Fecha de ejecución: 2026-07-30


Las tres especies objetivo están declaradas en un solo lugar,
`core/ingesta.py`, y de ahí las consumen tanto este notebook como la
aplicación. Así no hay dos listas que puedan desincronizarse.

In [2]:
from core.ingesta import ESPECIES_OBJETIVO

objetivo = pd.DataFrame(ESPECIES_OBJETIVO).T[
    ["id", "speciesKey", "nombre_comun", "nombre_cientifico", "impacto_ambiental", "autoridad"]
]
objetivo

,id,speciesKey,nombre_comun,nombre_cientifico,impacto_ambiental,autoridad
jabali,1,7705930,Jabali,Sus scrofa,Alto,SAG
liebre,2,7952072,Liebre europea,Lepus europaeus,Medio,SAG
rata gris,3,2439261,Rata gris,Rattus norvegicus,Alto,SAG


---
# Paso 1 — Descargar los registros de GBIF

Pedimos a GBIF **todas** las ocurrencias chilenas con coordenada de las tres
especies. Dos decisiones de diseño:

**Filtramos en origen, no después.** La consulta lleva `country=CL` y
`hasCoordinate=true`. Es más barato que descargar todo y filtrar en pandas, y
—como se verá en el paso 2— explica por qué la limpieza posterior descarta tan
poco.

**Bajamos registros individuales, no conteos agregados.** La versión anterior
del proyecto usaba *facetas* de GBIF (solo totales por especie y zona) porque
con 812 especies × 30 zonas los registros individuales eran inmanejables. Con
tres especies son ~49.000 filas, que caben de sobra en memoria. Y esto es lo
que habilita el análisis territorial: para preguntar por latitud, entorno
urbano/rural o evolución temporal hacen falta los puntos, no los totales.

La descarga queda en caché en `data/crudo/`, porque son ~165 peticiones para la
liebre y el notebook se re-ejecuta muchas veces.

In [3]:
from core.ingesta import descargar_todas

crudo = descargar_todas()          # usa la caché de data/crudo/ si ya existe
print(f"{len(crudo):,} registros crudos\n")
crudo.groupby("clase_modelo").size().rename("registros").to_frame()

49,165 registros crudos



,registros
clase_modelo,
jabali,3156
liebre,45778
rata gris,231


### Primera lectura: el desbalance

Antes de limpiar nada, un hecho que va a condicionar todo el análisis: las tres
especies **no** están representadas ni de cerca por igual.

In [4]:
conteo = crudo.groupby("clase_modelo").size().sort_values(ascending=False)
for clase, n in conteo.items():
    print(f"{clase:12s} {n:7,}  {100*n/len(crudo):5.1f}%   {'█' * int(60*n/conteo.max())}")
print(f"\nLa liebre tiene {conteo.iloc[0]/conteo.iloc[-1]:.0f} veces más registros que la rata gris.")

liebre        45,778   93.1%   ████████████████████████████████████████████████████████████
jabali         3,156    6.4%   ████
rata gris        231    0.5%   

La liebre tiene 198 veces más registros que la rata gris.


Una razón de 198 a 1 entre la especie más y menos registrada no es un detalle
menor. Cualquier conclusión del tipo "la liebre es la más extendida" tiene que
justificar primero que no está midiendo simplemente **quién fue observado más**.
Volvemos sobre esto en el paso 4, que resulta ser el hallazgo central.

---
# Paso 2 — Limpieza

La función `limpiar()` aplica los filtros y devuelve además una **bitácora**:
cuántas filas cayó cada paso y por qué. La idea es que la limpieza sea
auditable en vez de una caja negra.

Los pasos son: duplicados por `gbif_id`, coordenadas ausentes o no numéricas,
puntos fuera del rectángulo de Chile, la coordenada (0,0) —error clásico de
digitación—, y registros de *ausencia* (GBIF permite documentar que una especie
**no** estaba en un lugar, y eso no es un avistamiento).

In [5]:
from core.ingesta import limpiar

df, bitacora = limpiar(crudo)
bitacora

,paso,descartados,quedan
0,duplicados por gbif_id,0,49165
1,sin coordenada valida,0,49165
2,fuera del bounding box de Chile,0,49165
3,"coordenada en (0,0)",0,49165
4,registros de ausencia,0,49165
5,"sin fecha exacta (se marcan, no se descartan)",32,49165
6,TOTAL,0,49165


### La limpieza no descartó nada, y eso es un resultado

Todos los pasos marcan 0 descartes. No es que los filtros estén mal escritos:
es la consecuencia de haber **filtrado en origen**. `hasCoordinate=true` ya
garantiza coordenada válida, `country=CL` ya garantiza que el punto cae en
Chile, y GBIF deduplica por `gbif_id` antes de servir los datos.

Dejamos los filtros en el pipeline igualmente, por dos razones: documentan qué
supuestos estamos asumiendo sobre los datos, y si mañana alguien cambia la
consulta de la API o suma otra fuente, la bitácora avisará en vez de dejar
pasar basura en silencio.

Lo único que sí aparece son **32 registros sin fecha exacta** (traen solo el año
o el año-mes). No los botamos: su año sirve para la serie temporal, así que los
marcamos con `tiene_fecha` y quedan disponibles para todo lo demás.

In [6]:
print("Registros sin fecha exacta:", int((~df["tiene_fecha"]).sum()))
print("Formatos de eventDate encontrados (por largo del string):")
print(crudo["eventDate"].dropna().astype(str).str.len().value_counts().sort_index().to_string())

Registros sin fecha exacta: 32
Formatos de eventDate encontrados (por largo del string):
eventDate
4        16
7         1
10    48810
16      198
17        2
19      123


> **Nota de implementación que costó un bug.** `eventDate` llega en formatos
> mezclados: `2026-01-28`, `2026-01-28T07:03`, `2026-03-23T11:57:51` y a veces
> solo `2026`. Si se le pasa la columna entera a `pd.to_datetime(...,
> errors="coerce")`, pandas **infiere un único formato a partir del primer
> valor** y convierte a nulo todo lo que no calce. En nuestro caso el primer
> valor tenía hora, así que se perdían los 48.810 registros que traen solo
> `YYYY-MM-DD`: quedaban 198 de 49.165. La solución es recortar los primeros 10
> caracteres —la parte ISO común a todas las variantes— y parsear con formato
> explícito.

---
# Paso 3 — De coordenada a territorio

GBIF entrega coordenadas. Necesitamos región, comuna y saber si el punto está
en una ciudad.

**Por qué no usamos el campo `stateProvince` de GBIF:** viene sucio y en
formatos mezclados (`Region Metropolitana`, `RM`, `Santiago`, vacío). La
coordenada es el dato duro del registro, así que derivamos todo de ella.

**Comuna y región** salen del centro comunal más cercano, usando el diccionario
de `core/comunas.py` (346 comunas con coordenadas). Es una aproximación:
usamos el *centro* de la comuna, no su polígono real.

**Zona urbana** mantiene la definición del proyecto: 30 zonas urbanas definidas
como centro + radio en km. Un registro fuera de todos los radios es *rural*.

In [7]:
from core.ingesta import asignar_comuna, asignar_zona_urbana

zonas = pd.read_csv("data/zonas_urbanas.csv")
df = asignar_comuna(df)
df = asignar_zona_urbana(df, zonas)

print(df["entorno"].value_counts().to_string())
print(f"\n{100*(df['entorno']=='Urbano').mean():.1f}% de los registros cae en zona urbana")

entorno
Rural     47453
Urbano     1712

3.5% de los registros cae en zona urbana


### Hasta dónde podemos confiar en la comuna asignada

Aquí hay que ser explícitos sobre una debilidad. Si medimos a qué distancia
quedó cada registro del centro comunal que le asignamos:

In [8]:
d = df["dist_centro_comunal_km"]
print(d.describe([.25, .5, .75, .9, .99]).round(1).to_string())
print(f"\nRegistros a más de 25 km del centro comunal asignado: "
      f"{(~df['comuna_confiable']).sum():,} ({100*(~df['comuna_confiable']).mean():.0f}%)")

count    49165.0
mean        31.3
std         13.2
min          0.0
25%         21.4
50%         34.4
75%         39.4
90%         48.0
99%         59.5
max        617.3

Registros a más de 25 km del centro comunal asignado: 29,841 (61%)


La mediana es de ~34 km. Eso pasa porque el 96% de los registros son rurales y
las comunas del sur son enormes: un punto en medio de Aysén puede estar a 60 km
del pueblo más cercano y aun así ser "su" comuna.

**Consecuencia metodológica:** la columna `comuna` sirve para mostrar una
referencia legible en la app ("cerca de Cochamó"), pero **no** para agregar
resultados. El análisis del notebook 02 trabaja a nivel de **región** y de
**zona urbana**, que sí son robustos. Los registros con comuna dudosa quedan
marcados en `comuna_confiable`.

---
# Paso 4 — El hallazgo: no todos los registros se generaron igual

Este es el paso más importante del notebook, y no estaba planificado. Apareció
al revisar de dónde vienen los datos.

Miremos `basisOfRecord`, el campo que dice **cómo** se generó cada registro:

In [9]:
print(crudo["basisOfRecord"].value_counts().to_string())

basisOfRecord
MACHINE_OBSERVATION    48575
HUMAN_OBSERVATION        386
PRESERVED_SPECIMEN       182
MATERIAL_SAMPLE           20
MATERIAL_CITATION          2


El 98,8% son `MACHINE_OBSERVATION` — observación automatizada, no una persona
mirando un animal. Eso es inesperado para datos de biodiversidad, así que
conviene ver quién los publica:

In [10]:
print(crudo["institutionCode"].fillna("(vacío)").value_counts().head(6).to_string())

institutionCode
Corporación Nacional Forestal (CONAF)                            48303
iNaturalist                                                        336
MMA:GEFMONT                                                        254
MSB                                                                 96
CNX                                                                 51
Museo de Zoología de la Universidad de Concepción (MZUC-UCCC)       49


**48.303 de 49.165 registros (98,2%) son de CONAF**, la Corporación Nacional
Forestal, como observación automatizada. Es un programa de **cámaras trampa en
áreas silvestres protegidas**.

Esto reordena el análisis completo. Clasificamos cada registro según *cómo* se
generó, no según quién lo publica:

In [11]:
from core.ingesta import clasificar_fuente

df["fuente"] = clasificar_fuente(df)
pd.crosstab(df["fuente"], df["clase_modelo"], margins=True, margins_name="Total")

clase_modelo,jabali,liebre,rata gris,Total
fuente,,,,
Camara trampa (CONAF),3114,45189,0,48303
Ciencia ciudadana (iNaturalist),37,256,43,336
Coleccion museologica,0,31,173,204
Otros estudios,5,302,15,322
Total,3156,45778,231,49165


Y ahora la tabla que ordena el resto del análisis: **el mismo indicador, calculado
sobre la misma especie, cambia radicalmente según la fuente que lo registró.**

In [12]:
comp = df.pivot_table(index="fuente", columns="clase_modelo", values="entorno",
                      aggfunc=lambda s: round(100 * (s == "Urbano").mean(), 1))
print("PORCENTAJE DE REGISTROS EN ZONA URBANA (%)\n")
print(comp.to_string())
print("\n\nLATITUD MEDIANA\n")
print(df.pivot_table(index="fuente", columns="clase_modelo", values="lat",
                     aggfunc="median").round(2).to_string())

PORCENTAJE DE REGISTROS EN ZONA URBANA (%)

clase_modelo                     jabali  liebre  rata gris
fuente                                                    
Camara trampa (CONAF)               0.0     3.5        NaN
Ciencia ciudadana (iNaturalist)    13.5     8.6       55.8
Coleccion museologica               NaN     6.5       34.7
Otros estudios                     20.0     1.0        0.0


LATITUD MEDIANA

clase_modelo                     jabali  liebre  rata gris
fuente                                                    
Camara trampa (CONAF)            -41.06  -45.97        NaN
Ciencia ciudadana (iNaturalist)  -39.40  -47.11     -33.79
Coleccion museologica               NaN  -32.65     -36.54
Otros estudios                   -44.37  -33.65     -22.43


### Qué significa esto

Tres lecturas concretas:

1. **CONAF no tiene ni un registro de rata gris.** Cero de 48.303. No porque no
   haya ratas en Chile, sino porque no se instalan cámaras trampa en ciudades.
   La especie más urbana de las tres es invisible para la fuente que aporta el
   98% de los datos.

2. **El jabalí pasa de 0,0% urbano a 13,5%** según si lo miran las cámaras de
   CONAF o la gente en iNaturalist. Un mismo animal, dos números
   irreconciliables.

3. **La latitud mediana de la liebre es −45,97** en los datos de CONAF, que es
   Aysén. Eso no dice que la liebre viva preferentemente en Aysén: dice que
   **CONAF tiene muchas cámaras en Aysén**.

Es un caso de libro de **sesgo de muestreo**: el patrón que uno mediría al
mezclar las fuentes describe *dónde se observó*, no *dónde está la especie*.
Confundir ambas cosas es el error que este paso evita.

**Cómo lo tratamos.** `fuente` pasa a ser una columna de primera clase. El
notebook 02 reporta cada resultado territorial **desagregado por fuente**, y la
app deja filtrar por ella con una advertencia visible. No corregimos el sesgo
—no se puede con estos datos—, pero lo hacemos imposible de ignorar.

---
# Paso 5 — Fotos con licencia libre

Cada registro de GBIF puede traer imágenes. Nos sirven para ilustrar el
catálogo de la app con fotos reales tomadas en Chile.

Priorizamos por licencia: **CC0** (dominio público) primero, luego **CC-BY**, y
**CC-BY-NC** al final, porque restringe el uso comercial.

In [13]:
from core.ingesta import fotos_libres

fotos = fotos_libres(df, por_especie=12)
print(f"{len(fotos)} fotos seleccionadas\n")
print(fotos["license"].value_counts().to_string())
print("\nFoto de portada de cada especie (la de mejor licencia):")
fotos.groupby("nombre_comun").first()[["anio", "region", "license"]]

36 fotos seleccionadas

license
http://creativecommons.org/licenses/by/4.0/legalcode          22
http://creativecommons.org/publicdomain/zero/1.0/legalcode     8
http://creativecommons.org/licenses/by-nc/4.0/legalcode        6

Foto de portada de cada especie (la de mejor licencia):


,anio,region,license
nombre_comun,,,
Jabali,2024.0,Araucania,http://creativecommons.org/publicdomain/zero/1...
Liebre europea,2026.0,Magallanes,http://creativecommons.org/publicdomain/zero/1...
Rata gris,2024.0,Metropolitana,http://creativecommons.org/publicdomain/zero/1...


---
# Paso 6 — Guardar los datos procesados

`ejecutar_ingesta()` corre el pipeline completo y deja las tablas en
`data/procesado/`. Se puede llamar de una sola vez; los pasos anteriores fueron
para mostrar qué hace por dentro.

In [14]:
from core.ingesta import ejecutar_ingesta

salidas = ejecutar_ingesta()

1/5 Descargando ocurrencias de GBIF (con cache en data/crudo/)…
     49,165 registros crudos de 3 especies
2/5 Limpiando…
     49,165 registros limpios (0 descartados)
3/5 Asignando comuna, region y zona urbana…


     1,712 urbanos / 47,453 rurales
4/5 Construyendo tablas de analisis…
5/5 Guardando en data/procesado/…


     ocurrencias.csv  (49,165 filas)
     limpieza_bitacora.csv  (7 filas)
     especies_objetivo.csv  (3 filas)
     resumen_especie.csv  (3 filas)
     resumen_fuente.csv  (10 filas)
     resumen_region.csv  (15 filas)
     resumen_zona.csv  (30 filas)
     serie_temporal.csv  (30 filas)
     fotos_muestra.csv  (36 filas)


La publicación de los datos que consume la aplicación es un paso **aparte** a
propósito: la ingesta es análisis, y no debería sobrescribir en silencio los
archivos que lee la app.

In [15]:
from core.ingesta import publicar_para_app

publicar_para_app(salidas["ocurrencias"])

     especies.csv        (3 filas)
     32 registros sin fecha exacta quedan fuera del mapa (0.07% del total)
     avistamientos.csv   (49,133 filas)


---
# Resumen del paso de obtención y limpieza

| # | Problema encontrado | Cómo lo resolvimos |
|---|---|---|
| 1 | `eventDate` en 5 formatos; pandas infiere uno y descarta el resto | Recorte a los 10 primeros caracteres + formato explícito (rescató 48.935 registros) |
| 2 | `stateProvince` de GBIF sucio e inconsistente | No lo usamos: derivamos región y comuna de la coordenada |
| 3 | Comunas enormes en el sur: la comuna asignada queda a 34 km de mediana | La marcamos como poco confiable y agregamos por región, no por comuna |
| 4 | Registros de ausencia contarían como avistamientos | Filtrados por `occurrenceStatus` |
| 5 | **El 98% de los datos son cámaras trampa de CONAF en áreas protegidas** | **`fuente` como dimensión de primera clase; todo resultado territorial se reporta desagregado** |
| 6 | Desbalance de 198:1 entre liebre y rata gris | Se reporta siempre junto al indicador; nunca se comparan volúmenes absolutos entre especies |
| 7 | Territorio insular distorsionaría el análisis latitudinal | Marcado en `territorio` (resultó ser 1 solo registro) |

## Lo que queda listo para el análisis

`data/procesado/ocurrencias.csv` — 49.165 registros de las tres especies, con
región, comuna, zona urbana, entorno, fuente y fecha normalizada. Más las
tablas agregadas (`resumen_especie`, `resumen_fuente`, `resumen_region`,
`resumen_zona`, `serie_temporal`) y la muestra de fotos.

El notebook **02_analisis_territorial** responde la pregunta con estos datos.